# Phase 2 – Data Understanding

**Flash Eurobarometer 564 — SMEs, Resource Efficiency & Green Transition**  
CRISP-DM Phase 2: Data Understanding

In [ ]:
import sys
sys.path.insert(0, '..')

import pyreadstat
import pandas as pd
import numpy as np
import matplotlib
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import warnings
warnings.filterwarnings('ignore')

%matplotlib inline

BLUE       = '#1a3a5c'
LIGHT_BLUE = '#4a90c4'
RED        = '#c0392b'
AMBER      = '#d4821a'
BG         = '#f8f9fa'
WIDTH      = 0.35

## 0. Load Data

In [ ]:
df, meta = pyreadstat.read_sav('../data/raw/initial_data.sav', apply_value_formats=False)
df_labeled, _ = pyreadstat.read_sav('../data/raw/initial_data.sav', apply_value_formats=True)

pt = df[df['isocntry'] == 'PT'].copy()
print(f"Total rows: {len(df):,} | Portugal: {len(pt):,}")

## 1. Shape & Structure

In [ ]:
print("=== SHAPE ===")
print(f"Rows:    {df.shape[0]:,}")
print(f"Columns: {df.shape[1]}")

print("\n=== DATA TYPES ===")
print(df.dtypes.value_counts())

print("\n=== ALL VARIABLES & LABELS ===")
for col in df.columns:
    label = meta.column_names_to_labels.get(col, '')
    print(f"  {col}: {label}")

## 2. Sample Composition

In [ ]:
print("=== COUNTRY DISTRIBUTION ===")
print(df['isocntry'].value_counts())

In [ ]:
sector_labels = {1.0: 'Manufacturing (C)', 2.0: 'Industry (B/D/E/F)',
                 3.0: 'Retail (G)', 4.0: 'Services (H-M)'}

print("=== SECTOR DISTRIBUTION (nace_b) ===")
print("EU:")
print(df['nace_b'].map(sector_labels).value_counts(normalize=True).round(3) * 100)
print("PT:")
print(pt['nace_b'].map(sector_labels).value_counts(normalize=True).round(3) * 100)

In [ ]:
size_labels = {1.0: 'Micro (1-9)', 2.0: 'Small (10-49)', 3.0: 'Medium (50-249)',
               4.0: 'Large (250-499)', 5.0: 'Very large (500+)', 6.0: 'DK/NA'}

print("=== FIRM SIZE (scr10) ===")
print("EU:")
print(df['scr10'].map(size_labels).value_counts(normalize=True).round(3) * 100)
print("PT:")
print(pt['scr10'].map(size_labels).value_counts(normalize=True).round(3) * 100)

print("\n=== COMPANY TYPE (d1b) ===")
print(df_labeled['d1b'].value_counts())

## 3. Value Labels for Key Variables

In [ ]:
key_vars = ['d1b', 'scr10', 'scr10b', 'scr13a', 'scr14', 'nace_a', 'nace_b',
            'q3', 'q4', 'n1a', 'n1b', 'q9', 'q14', 'n2', 'n3']

for var in key_vars:
    if var in meta.variable_value_labels:
        print(f"\n{var}: {meta.column_names_to_labels.get(var, '')}")
        for k, v in meta.variable_value_labels[var].items():
            print(f"  {k} = {v}")

## 4. Missing Data

In [ ]:
print("=== MISSING DATA (all variables with any missing) ===")
missing = df.isnull().sum()
missing_pct = (missing / len(df) * 100).round(1)
missing_df = pd.DataFrame({'missing_n': missing, 'missing_pct': missing_pct})
print(missing_df[missing_df['missing_n'] > 0].sort_values('missing_pct', ascending=False))

print("\n=== MISSING DATA - SUBSTANTIVE VARS ONLY ===")
core_vars = ['nace_b', 'scr10', 'scr13a', 'scr14', 'q3', 'q4', 'n1a', 'q9', 'q14']
for v in core_vars:
    m = df[v].isnull().sum()
    print(f"  {v}: {m} ({m / len(df) * 100:.1f}%)")

## 5. Key Variable Distributions

In [ ]:
turn_map = {1: '<25k', 2: '25k-50k', 3: '50k-100k', 4: '100k-250k', 5: '250k-500k',
            6: '500k-2M', 7: '2M-10M', 8: '10M-50M', 9: '>50M', 10: 'DK/NA'}

print("=== TURNOVER 2023 (scr14) ===")
print(df['scr14'].map(turn_map).value_counts(normalize=True).round(3) * 100)

trend_map = {1: 'Increased 10%+', 2: 'Increased <10%', 3: 'Unchanged', 4: 'Decreased', 998: 'DK'}
print("\n=== TURNOVER TREND (scr13a) EU vs PT ===")
print("EU:", df['scr13a'].map(trend_map).value_counts(normalize=True).round(3).to_dict())
print("PT:", pt['scr13a'].map(trend_map).value_counts(normalize=True).round(3).to_dict())

In [ ]:
n1a_map = {1: 'Yes - purchase', 2: 'Yes - generate onsite', 3: 'No', 4: 'DK'}
print("=== RENEWABLE ENERGY (n1a) EU vs PT ===")
print("EU:", df['n1a'].map(n1a_map).value_counts(normalize=True).round(3).to_dict())
print("PT:", pt['n1a'].map(n1a_map).value_counts(normalize=True).round(3).to_dict())

q3_map = {1: 'Sig. decreased', 2: 'Slightly decreased', 3: 'Slightly increased',
          4: 'Sig. increased', 5: 'No change', 6: 'DK', 9: 'N/A'}
q3_valid = df[df['q3'].isin([1, 2, 3, 4, 5])]
q3_pt    = pt[pt['q3'].isin([1, 2, 3, 4, 5])]
print("\n=== COST IMPACT OF RE ACTIONS (q3) EU vs PT ===")
print("EU:", q3_valid['q3'].map(q3_map).value_counts(normalize=True).round(3).to_dict())
print("PT:", q3_pt['q3'].map(q3_map).value_counts(normalize=True).round(3).to_dict())

dx5r_map = {1: 'None', 2: 'Less than half', 3: 'About half', 4: 'More than half', 5: 'All'}
dx5r_eu = df[df['dx5r'].isin([1,2,3,4,5])]['dx5r'].map(dx5r_map).value_counts(normalize=True) * 100
dx5r_pt = pt[pt['dx5r'].isin([1,2,3,4,5])]['dx5r'].map(dx5r_map).value_counts(normalize=True) * 100
print("\n=== GREEN JOBS (dx5r) EU vs PT ===")
print("EU:", dx5r_eu.round(1).to_dict())
print("PT:", dx5r_pt.round(1).to_dict())

## 6. Resource Efficiency Actions (Q1)

In [ ]:
q1_map = {
    'q1.1': 'Saving water',
    'q1.2': 'Saving energy',
    'q1.3': 'Using renewable energy',
    'q1.4': 'Saving materials',
    'q1.5': 'Switching to greener suppliers',
    'q1.6': 'Minimising waste',
    'q1.7': 'Selling residues/waste',
    'q1.8': 'Recycling within company',
    'q1.9': 'Designing for repair/reuse',
}

print("=== Q1: CURRENT RESOURCE EFFICIENCY ACTIONS (% selecting) ===")
print(f"{'Action':<35} {'EU %':>8} {'PT %':>8}")
print("-" * 55)
for col, label in q1_map.items():
    eu_pct = (df[col] == 1).mean() * 100
    pt_pct = (pt[col] == 1).mean() * 100
    print(f"{label:<35} {eu_pct:>7.1f}% {pt_pct:>7.1f}%")

## 7. Barriers (Q7)

In [ ]:
q7_map = {
    'q7.1':  'Lack of finance/investment',
    'q7.2':  'High upfront cost',
    'q7.3':  'Lack of technical expertise',
    'q7.4':  'Lack of time/staff',
    'q7.5':  'Lack of information',
    'q7.6':  'Too much admin/complexity',
    'q7.7':  'Regulations/incentives absent',
    'q7.8':  'No support available',
    'q7.9':  'Uncertain ROI',
    'q7.10': "Customers don't demand it",
    'q7.11': 'Supplier constraints',
    'q7.12': 'Lack of suitable tech',
    'q7.13': 'None',
}

print("=== Q7: BARRIERS (% reporting) ===")
print(f"{'Barrier':<35} {'EU %':>8} {'PT %':>8}")
print("-" * 55)
for col, label in q7_map.items():
    if col in df.columns:
        eu_pct = (df[col] == 1).mean() * 100
        pt_pct = (pt[col] == 1).mean() * 100
        print(f"{label:<35} {eu_pct:>7.1f}% {pt_pct:>7.1f}%")

## 8. Weighting System

In [ ]:
weight_cols = [c for c in df.columns if c.startswith('w')]
print(f"Total weight variables: {len(weight_cols)}")
print("\nw1 (post-strat, all companies):")
print(df['w1'].describe().round(3))
print("\nw1_sme (post-strat, SMEs only):")
print(df['w1_sme'].describe().round(3))

## 9. Visualisations

### Fig 1: Country sample sizes

In [ ]:
country_counts = df['isocntry'].value_counts().sort_values()
colors = [RED if c == 'PT' else (BLUE if c in ['DE', 'FR', 'IT', 'ES', 'PL'] else LIGHT_BLUE)
          for c in country_counts.index]

fig, ax = plt.subplots(figsize=(12, 6), facecolor=BG)
ax.barh(country_counts.index, country_counts.values, color=colors, edgecolor='white', linewidth=0.5)
ax.set_xlabel('Number of respondents', fontsize=11)
ax.set_title('Sample size by country', fontsize=14, fontweight='bold', pad=15)
ax.set_facecolor(BG)
fig.patch.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
ax.legend(handles=[
    mpatches.Patch(color=RED, label='Portugal'),
    mpatches.Patch(color=BLUE, label='5 largest EU economies'),
    mpatches.Patch(color=LIGHT_BLUE, label='Other'),
], fontsize=9, loc='lower right')
plt.tight_layout()

### Fig 2: Sector distribution EU vs Portugal

In [ ]:
eu_sector = df['nace_b'].value_counts(normalize=True).sort_index() * 100
pt_sector = pt['nace_b'].value_counts(normalize=True).sort_index() * 100
x = np.arange(len(sector_labels))

fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.bar(x - WIDTH/2, [eu_sector.get(k, 0) for k in sorted(sector_labels)], WIDTH, label='EU', color=BLUE)
ax.bar(x + WIDTH/2, [pt_sector.get(k, 0) for k in sorted(sector_labels)], WIDTH, label='Portugal', color=RED)
ax.set_xticks(x)
ax.set_xticklabels([sector_labels[k] for k in sorted(sector_labels)], fontsize=10)
ax.set_ylabel('Share (%)')
ax.set_title('Sector distribution: EU vs Portugal', fontsize=14, fontweight='bold', pad=15)
ax.legend()
ax.set_facecolor(BG); fig.patch.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

### Fig 3: Firm size EU vs Portugal

In [ ]:
categories = sorted(size_labels.keys())
eu_size = df['scr10'].value_counts(normalize=True).sort_index() * 100
pt_size = pt['scr10'].value_counts(normalize=True).sort_index() * 100
x = np.arange(len(categories))

fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
ax.bar(x - WIDTH/2, [eu_size.get(k, 0) for k in categories], WIDTH, label='EU', color=BLUE)
ax.bar(x + WIDTH/2, [pt_size.get(k, 0) for k in categories], WIDTH, label='Portugal', color=RED)
ax.set_xticks(x)
ax.set_xticklabels([size_labels[k] for k in categories], fontsize=10)
ax.set_ylabel('Share (%)')
ax.set_title('Firm size (FTE): EU vs Portugal', fontsize=14, fontweight='bold', pad=15)
ax.legend()
ax.set_facecolor(BG); fig.patch.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

### Fig 4: Q1 Resource efficiency actions

In [ ]:
eu_q1 = {label: (df[col] == 1).mean() * 100 for col, label in q1_map.items()}
pt_q1 = {label: (pt[col] == 1).mean() * 100 for col, label in q1_map.items()}
sorted_items  = sorted(eu_q1.items(), key=lambda x: x[1])
labels_sorted = [i[0] for i in sorted_items]
eu_vals = [i[1] for i in sorted_items]
pt_vals = [pt_q1[l] for l in labels_sorted]

fig, ax = plt.subplots(figsize=(10, 7), facecolor=BG)
y = np.arange(len(labels_sorted))
ax.barh(y - 0.2, eu_vals, 0.4, label='EU', color=BLUE)
ax.barh(y + 0.2, pt_vals, 0.4, label='Portugal', color=RED)
ax.set_yticks(y)
ax.set_yticklabels(labels_sorted, fontsize=10)
ax.set_xlabel('% of firms')
ax.set_title('Current resource efficiency actions (Q1)', fontsize=14, fontweight='bold', pad=15)
ax.legend()
ax.set_facecolor(BG); fig.patch.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

### Fig 5: Q7 Barriers

In [ ]:
eu_q7 = {label: (df[col] == 1).mean() * 100 for col, label in q7_map.items() if col in df.columns}
pt_q7 = {label: (pt[col] == 1).mean() * 100 for col, label in q7_map.items() if col in df.columns}
sorted_items  = sorted(eu_q7.items(), key=lambda x: x[1])
labels_sorted = [i[0] for i in sorted_items]
eu_vals = [i[1] for i in sorted_items]
pt_vals = [pt_q7[l] for l in labels_sorted]

fig, ax = plt.subplots(figsize=(10, 7), facecolor=BG)
y = np.arange(len(labels_sorted))
ax.barh(y - 0.2, eu_vals, 0.4, label='EU', color=BLUE)
ax.barh(y + 0.2, pt_vals, 0.4, label='Portugal', color=RED)
ax.set_yticks(y)
ax.set_yticklabels(labels_sorted, fontsize=10)
ax.set_xlabel('% of firms')
ax.set_title('Barriers to resource efficiency (Q7)', fontsize=14, fontweight='bold', pad=15)
ax.legend()
ax.set_facecolor(BG); fig.patch.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
plt.tight_layout()

### Fig 6: Missing data

In [ ]:
key_cols = (list(q1_map.keys()) + [f'q7.{i}' for i in range(1, 14)] +
            ['q3', 'q4', 'n1a', 'n1b', 'q9', 'q14', 'q10', 'n2', 'n3',
             'scr10', 'scr13a', 'scr14', 'nace_b'])
miss_pct = df[key_cols].isnull().mean() * 100

fig, ax = plt.subplots(figsize=(12, 4), facecolor=BG)
miss_pct.sort_values(ascending=False).plot(kind='bar', ax=ax, color=BLUE, edgecolor='white')
ax.set_ylabel('Missing %')
ax.set_title('Missing data by substantive variable', fontsize=14, fontweight='bold', pad=15)
ax.set_facecolor(BG); fig.patch.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(axis='x', rotation=90, labelsize=8)
plt.tight_layout()

### Fig 7: Green products & carbon strategy

In [ ]:
q9_labels  = {1.0: 'Yes', 2.0: 'Plan to', 3.0: 'No', 4.0: 'DK/NA'}
q14_labels = {1.0: 'Yes', 2.0: 'Plan to', 3.0: 'No', 4.0: 'Already neutral', 5.0: 'DK/NA'}

fig, axes = plt.subplots(1, 2, figsize=(12, 5), facecolor=BG)
for ax, col, lmap, title in zip(
    axes, ['q9', 'q14'], [q9_labels, q14_labels],
    ['Green products/services (Q9)', 'Carbon neutrality strategy (Q14)']
):
    eu_v = df[col].value_counts(normalize=True).sort_index() * 100
    pt_v = pt[col].value_counts(normalize=True).sort_index() * 100
    cats = sorted(lmap.keys())
    x = np.arange(len(cats))
    ax.bar(x - 0.2, [eu_v.get(k, 0) for k in cats], 0.4, label='EU', color=BLUE)
    ax.bar(x + 0.2, [pt_v.get(k, 0) for k in cats], 0.4, label='Portugal', color=RED)
    ax.set_xticks(x)
    ax.set_xticklabels([lmap[k] for k in cats], fontsize=9, rotation=15, ha='right')
    ax.set_ylabel('Share (%)')
    ax.set_title(title, fontsize=12, fontweight='bold')
    ax.legend(fontsize=9)
    ax.set_facecolor(BG)
    ax.spines[['top', 'right']].set_visible(False)
fig.patch.set_facecolor(BG)
plt.tight_layout()

### Fig 8: Investment level by firm size

In [ ]:
q4_labels = {1.0: 'Nothing', 2.0: '<1%', 3.0: '1-5%', 4.0: '6-10%',
             5.0: '11-30%', 6.0: '>30%'}

q4_sub = df[df['q4'].isin([1, 2, 3, 4, 5, 6])].copy()
q4_sub['q4_label'] = q4_sub['q4'].map(q4_labels)
q4_sub['size']     = q4_sub['scr10'].map({1.0: '1-9', 2.0: '10-49', 3.0: '50-249'})
q4_sub = q4_sub.dropna(subset=['size'])

size_q4 = q4_sub.groupby(['size', 'q4_label']).size().unstack(fill_value=0)
size_q4 = size_q4.div(size_q4.sum(axis=1), axis=0) * 100
order_cats = ['Nothing', '<1%', '1-5%', '6-10%', '11-30%', '>30%']
size_q4 = size_q4[[c for c in order_cats if c in size_q4.columns]]

fig, ax = plt.subplots(figsize=(10, 5), facecolor=BG)
size_q4.plot(kind='bar', ax=ax, colormap='Blues', edgecolor='white')
ax.set_xlabel('Firm size (FTE)')
ax.set_ylabel('Share (%)')
ax.set_title('Investment in resource efficiency by firm size (Q4)', fontsize=14, fontweight='bold', pad=15)
ax.legend(title='Investment level', fontsize=9, bbox_to_anchor=(1, 1))
ax.set_facecolor(BG); fig.patch.set_facecolor(BG)
ax.spines[['top', 'right']].set_visible(False)
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()